# Comparación de modelos de machine learning para la predicción de cáncer de pulmón

## Importación de librerías y modelos

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna

from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10,6)

print("Librerías importadas correctamente")

c:\Users\Baku\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Librerías importadas correctamente


## Dataset

In [ ]:
DATA_PATH = "data/dataset.csv"

df = pd.read_csv(DATA_PATH, sep=";")
print(f"Cantidad de datos: {df.shape[0]}")
print(df.head())

Cantidad de datos: 3309
  GENDER  AGE  SMOKING  YELLOW_FINGERS  ANXIETY  PEER_PRESSURE  \
0      M   65        1               1        1              2   
1      F   55        1               2        2              1   
2      F   78        2               2        1              1   
3      M   60        2               1        1              1   
4      F   80        1               1        2              1   

   CHRONIC_DISEASE  FATIGUE  ALLERGY  WHEEZING  ALCOHOL_CONSUMING  COUGHING  \
0                2        1        2         2                  2         2   
1                1        2        2         2                  1         1   
2                1        2        1         2                  1         1   
3                2        1        2         1                  1         2   
4                1        2        1         2                  1         1   

   SHORTNESS_OF_BREATH  SWALLOWING_DIFFICULTY  CHEST_PAIN LUNG_CANCER  
0                    2          

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3309 entries, 0 to 3308
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   GENDER                 3309 non-null   str  
 1   AGE                    3309 non-null   int64
 2   SMOKING                3309 non-null   int64
 3   YELLOW_FINGERS         3309 non-null   int64
 4   ANXIETY                3309 non-null   int64
 5   PEER_PRESSURE          3309 non-null   int64
 6   CHRONIC_DISEASE        3309 non-null   int64
 7   FATIGUE                3309 non-null   int64
 8   ALLERGY                3309 non-null   int64
 9   WHEEZING               3309 non-null   int64
 10  ALCOHOL_CONSUMING      3309 non-null   int64
 11  COUGHING               3309 non-null   int64
 12  SHORTNESS_OF_BREATH    3309 non-null   int64
 13  SWALLOWING_DIFFICULTY  3309 non-null   int64
 14  CHEST_PAIN             3309 non-null   int64
 15  LUNG_CANCER            3309 non-null   str  
dtyp

In [ ]:
# Revisar
binary_cols_csv = ['SMOKING', 'YELLOW_FINGERS', 'ANXIETY', 'FATIGUE', 
                   'WHEEZING', 'ALCOHOL_CONSUMING', 'COUGHING',     
                   'SHORTNESS_OF_BREATH', 'SWALLOWING_DIFFICULTY', 'CHEST_PAIN']   

for col in binary_cols_csv:
    if col in df.columns:
        df[col] = df[col].map({2:1,1:0}) 


df.head(5)

,GENDER,AGE,SMOKING,YELLOW_FINGERS,ANXIETY,PEER_PRESSURE,CHRONIC_DISEASE,FATIGUE,ALLERGY,WHEEZING,ALCOHOL_CONSUMING,COUGHING,SHORTNESS_OF_BREATH,SWALLOWING_DIFFICULTY,CHEST_PAIN,LUNG_CANCER
0,M,65,0,0,0,2,2,0,2,1,1,1,1,1,0,NO
1,F,55,0,1,1,1,1,1,2,1,0,0,0,1,1,NO
2,F,78,1,1,0,1,1,1,1,1,0,0,1,0,0,YES
3,M,60,1,0,0,1,2,0,2,0,0,1,0,1,1,YES
4,F,80,0,0,1,1,1,1,1,1,0,0,0,0,1,NO
